# Projet sujet n°4 : Gibbs et ABC
## Ina Campan, Gildas Montcel et Alistair Renaud 

Importation des librairies nécessaires

In [ ]:
# -q : quiet --> réduction des messages
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import invgamma, gaussian_kde, dirichlet
import time
import warnings
import math

warnings.filterwarnings('ignore')

In [ ]:
# paramétres globaux
SEED = 20798
T = 100
n = 5
alpha = [1., 2., 3.]
zeta = [1., 1.]

#N_tot normalement 1.1M
N_TOT = 1_100_000
N_KEEP = 1_000
np.random.seed()
rng_data = np.random.default_rng(SEED)

# Partie I : Implémentation de l’algorithme Gibbs-ABC
## Introduction

Dans cette partie , nous implémentons l'algorithme Gibbs-ABC proposé dans l’article *"Component-wise approximate Bayesian computation via Gibbs-like steps"*.

L’objectif est d’estimer les paramètres d’un modèle de séries temporelles de type Moving Average d’ordre 2 (MA(2)) à l’aide de méthodes d’inférence bayésienne approchée.

---

## Modèle MA(2)

On considère le modèle suivant :

$$
X_t = Y_t + \theta_1 Y_{t-1} + \theta_2 Y_{t-2}
$$

où :
- $(Y_t)$ est un bruit blanc gaussien tel que $Z_t \sim \mathcal{N}(0,1)$
- $\theta = (\theta_1, \theta_2)$ est le vecteur de paramètres à estimer

À partir de valeurs vraies $\theta_{\text{true}}$, nous simulons une série observée $x_{\text{obs}}$.

Le MA(2) est intéressant car il est simple à simuler, il reste non trivial, il introduit une dépendance temporelle de court terme, ce qui le rend pertinent pour l’étude de séries temporelles et enfin sa vraisemblance est difficile à manipuler analytiquement ce qui justifie l’usage de méthodes approchées comme l’ABC.
Ainsi, le modèle MA(2) est particulièrement adapté pour comparer et analyser les performances des algorithmes d’inférence bayésienne approchée.

---

## Approximate Bayesian Computation (ABC)

Dans ce modèle, la vraisemblance est difficile à exploiter directement.  
Les méthodes ABC permettent de contourner ce problème en procédant comme suit :

1. Simuler des données $x_{\text{sim}}$ à partir d’un paramètre $\theta$
2. Comparer des statistiques résumées $s(x_{\text{sim}})$ et $s(x_{\text{obs}})$
3. Accepter $\theta$ si la distance est suffisamment petite

Formellement, on accepte $\theta$ si :

$$
d\big(s(x_{\text{sim}}), s(x_{\text{obs}})\big) < \epsilon
$$

où $\epsilon > 0$ est un seuil de tolérance.

---

## Algorithme Gibbs-ABC

L’algorithme Gibbs-ABC améliore l’ABC standard en mettant à jour les paramètres un par un, de manière similaire à un échantillonneur de Gibbs.

À chaque itération, on effectue :

$$
\theta_1^{(t+1)} \sim \text{ABC}(\theta_1 \mid \theta_2^{(t)}, x_{\text{obs}})
$$

$$
\theta_2^{(t+1)} \sim \text{ABC}(\theta_2 \mid \theta_1^{(t+1)}, x_{\text{obs}})
$$

À chaque itération, les paramètres sont mis à jour un par un.
On commence par mettre à jour $\theta_1$ en gardant $\theta_2$ fixé, puis on met à jour $\theta_2$ en utilisant la nouvelle valeur de $\theta_1$.
Chaque mise à jour repose sur un critère ABC : on simule des données et on accepte la proposition si les statistiques simulées sont suffisamment proches des statistiques observées.

Cette approche permet de réduire le problème à 1 dimension seulement plutôt que de proposer tout $\theta$. On améliore ainsi l’efficacité de l’algorithme.

## Algorithme Gibbs-ABC (version hiérarchique)

Dans le modèle considéré du papier, les paramètres $\theta_j = (\theta_{j,1}, \theta_{j,2})$ de chaque série ne sont pas indépendants, mais issus d’une structure hiérarchique.

On introduit des variables latentes $\beta_j = (\beta_{j,1}, \beta_{j,2}, \beta_{j,3})$ telles que :
$$
\beta_j \sim \text{Dirichlet}(\alpha)
$$

Les paramètres du modèle MA(2) sont ensuite obtenus par transformation :
$$
\theta_{j,1} = \beta_{j,1} - \beta_{j,2}, \quad
\theta_{j,2} = 2(\beta_{j,1} + \beta_{j,2}) - 1
$$

Le paramètre $\alpha = (\alpha_1, \alpha_2, \alpha_3)$ est un hyperparamètre global qui contrôle la distribution des $\beta_j$, et donc la variabilité des $\theta_j$ entre les différentes séries.

La structure du modèle est donc la suivante :
$$
\alpha \;\rightarrow\; \beta_j \;\rightarrow\; \theta_j \;\rightarrow\; x_j
$$

L’objectif est d’estimer simultanément les paramètres locaux $\theta_j$ et le paramètre global $\alpha$ à partir des données observées $x_j$.

Comme la vraisemblance est intractable, on utilise une approche ABC combinée à un schéma de Gibbs. À chaque itération, on procède par mises à jour conditionnelles :

- **Mise à jour des $\theta_j$** :  
  pour chaque série $j$, on génère des candidats $\beta_j \sim \text{Dirichlet}(\alpha)$, que l’on transforme en $\theta_j$.  
  On simule ensuite des données et on sélectionne le paramètre qui produit des statistiques résumées proches des données observées.

- **Mise à jour de $\alpha$** :  
  on propose une nouvelle valeur de $\alpha$, puis on génère des $\beta_j$ et des $\theta_j$ associés.  
  La nouvelle valeur est retenue si elle permet de reproduire correctement la structure observée des paramètres.

Cette approche permet d’exploiter la structure hiérarchique du modèle tout en évitant le calcul explicite de la vraisemblance. Elle améliore également l’efficacité par rapport à un ABC standard en réduisant la dimension des mises à jour et en partageant l’information entre les séries via le paramètre $\alpha$.

## Simulation du Moving Average d'ordre 2

In [ ]:
def simulate_ma2(theta, sigma, T):
    y = np.random.normal(0, sigma, T + 2)
    x = np.zeros(T)
    for t in range(T):
        x[t] = y[t+2] + theta[0]*y[t+1] + theta[1]*y[t]
    return x

Commentaires : On simule une fonction qui génère une série temporelle. On récupère ensuite nos deux paramètres theta1 et theta2. On génère Y qui suit une loi normale centrée réduite. (n + 2) car MA(2) dépend de t-1 et de t-2. X est la série observée initialisée. Puis on simule la boucle temporelle, en commençant à 2 pour pouvoir utiliser t-1 et t-2. On trouve bien la relation suivante : 
$$
    X_t = Y_t + \theta_1 Y_{t-1} + \theta_2 Y_{t-2}
$$ 
On renvoie in fine la série simulée. 

## Autocorrélations

In [ ]:
def autocorr(x, lag):
    return np.corrcoef(x[:-lag], x[lag:])[0,1]

def summary_stats(x):
    return np.array([
        np.var(x),
        autocorr(x,1),
        autocorr(x,2)
    ])

Commentaires : 
Dans le cadre de l’ABC, si la vraisemblance du modèle est intractable, on remplace la comparaison directe des données par une comparaison de statistiques résumées. Le choix de ces statistiques est crucial : elles doivent capturer l’essentiel de la dépendance temporelle induite par les paramètres du modèle.

Pour un processus MA(2) défini par :
$$
X_t = Y_t + \theta_1 Y_{t-1} + \theta_2 Y_{t-2}, \quad Y_t \sim \mathcal{N}(0, \sigma^2)
$$

la structure de dépendance est entièrement caractérisée par les autocovariances aux retards 1 et 2 :

$$
\gamma(0) = \mathbb{E}[X_t^2] = \sigma^2 (1 + \theta_1^2 + \theta_2^2)
$$

$$
\gamma(1) = \mathbb{E}[X_t X_{t-1}] = \sigma^2 (\theta_1 + \theta_1 \theta_2)
$$

$$
\gamma(2) = \mathbb{E}[X_t X_{t-2}] = \sigma^2 \theta_2
$$

et pour tout retard $h > 2$ :
$$
\gamma(h) = \mathbb{E}[X_t X_{t-h}] = 0
$$

Les autocorrélations sont ensuite définies par :
$$
\rho(h) = \frac{\gamma(h)}{\gamma(0)}
$$

Ainsi, dans un modèle MA(2), les paramètres $\theta_1$ et $\theta_2$ déterminent directement les autocorrélations aux retards 1 et 2.

C’est pourquoi on utilise comme statistiques résumées :
- l’autocorrélation au lag 1
- l’autocorrélation au lag 2

Ces deux quantités suffisent à capturer l’essentiel de l’information sur les paramètres du modèle. Dans l’algorithme ABC, on compare donc les autocorrélations des données simulées à celles des données observées afin d’évaluer la qualité des paramètres proposés.

## Distance (pour ABC)

In [ ]:
def distance(s1, s2):
    return np.sum((s1 - s2)**2)

## Transformation $\beta$ → $\theta$

On veut tirer $\mu_j$ dans la région de stationnarité du MA(2) :

$$\mathcal{M}_2 = \{(\mu_1, \mu_2) : |\mu_2| < 1,\; \mu_2 + \mu_1 > -1,\; \mu_2 - \mu_1 > -1\}$$

Ces trois contraintes définissent un triangle dans $\mathbb{R}^2$. La stationnarité garantit que la série ne diverge pas et que ses autocorrélations sont bien définies.

On tire $(\beta_1, \beta_2, \beta_3) \sim \text{Dir}(\alpha)$ avec $\beta_1 + \beta_2 + \beta_3 = 1$, $\beta_k > 0$, puis :

$$\mu_1 = \beta_1 - \beta_2, \qquad \mu_2 = 2(\beta_1 + \beta_2) - 1$$

Les trois conditions de $\mathcal{M}_2$ sont automatiquement satisfaites pour tout tirage Dirichlet :
- $|\mu_2| < 1$ : car $\beta_3 > 0$ implique $0 < \beta_1+\beta_2 < 1$, donc $-1 < 2(\beta_1+\beta_2)-1 < 1$
- $\mu_2 + \mu_1 = 3\beta_1 + \beta_2 - 1 > -1$ car $\beta_1, \beta_2 > 0$
- $\mu_2 - \mu_1 = \beta_1 + 3\beta_2 - 1 > -1$ pour la même raison

C'est une bijection affine entre le simplexe et le triangle $\mathcal{M}_2$.

In [ ]:
def beta_to_theta(beta):
    b1, b2, b3 = beta
    theta1 = b1 - b2
    theta2 = 2*(b1 + b2) - 1
    return np.array([theta1, theta2])

Commentaires : 
Dans le modèle hiérarchique, les paramètres du MA(2) ne sont pas directement échantillonnés. On introduit des variables latentes $\beta = (\beta_1, \beta_2, \beta_3)$ suivant une loi de Dirichlet, puis on définit les paramètres $\theta$ comme une transformation déterministe de $\beta$.

Cette transformation, donnée par :
$$
\theta_1 = \beta_1 - \beta_2, \quad
\theta_2 = 2(\beta_1 + \beta_2) - 1
$$

est imposée par le papier. Elle permet de garantir que les paramètres $\theta$ restent dans une région admissible du modèle MA(2), tout en facilitant l’introduction d’une structure hiérarchique via la loi de Dirichlet.

Ainsi, on échantillonne $\beta$ puis on en déduit $\theta$, plutôt que d’échantillonner directement $\theta$.

## Mise à jour des paramètres $\theta_j$

In [ ]:
# On update les theta_j 
def update_theta_j(x_obs, alpha, sigma, T, N_theta):
    s_obs = summary_stats(x_obs)

    best_theta = None
    best_dist = np.inf

    for _ in range(N_theta):
        beta = dirichlet.rvs(alpha)[0]
        theta = beta_to_theta(beta)

        x_sim = simulate_ma2(theta, sigma, T)
        s_sim = summary_stats(x_sim)

        d = distance(s_sim, s_obs)

        if d < best_dist:
            best_dist = d
            best_theta = theta

    return best_theta

Commentaires : 
Cette étape correspond à la mise à jour des paramètres locaux $\theta_j$ pour chaque série, en utilisant une approche ABC conditionnelle.
Étant donné les données observées $x_{\text{obs}}$, on calcule leurs statistiques résumées (autocorrélations). On génère ensuite $N_{\theta}$ candidats en échantillonnant $\beta \sim \text{Dirichlet}(\alpha)$, puis en les transformant en $\theta$ via la relation imposée par le modèle. Pour chaque candidat, on simule une série temporelle et on calcule ses statistiques résumées.
On sélectionne finalement le paramètre $\theta$ qui minimise la distance entre les statistiques simulées et observées. Cette procédure correspond à une approximation de la distribution conditionnelle $\pi(\theta_j \mid \alpha, x_{\text{obs}})$ dans le cadre ABC, en retenant le candidat le plus compatible avec les données.

## Mise à jour du paramètre global $\alpha$

In [ ]:
# On update alpha
def update_alpha(thetas, N_alpha):
    best_alpha = None
    best_dist = np.inf

    # summary = moyenne des log-beta inverses (proxy)
    s_obs = np.mean(thetas, axis=0)

    for _ in range(N_alpha):
        alpha_candidate = np.random.exponential(1, size=3)

        beta_sim = dirichlet.rvs(alpha_candidate, size=len(thetas))
        theta_sim = np.array([beta_to_theta(b) for b in beta_sim])

        s_sim = np.mean(theta_sim, axis=0)

        d = np.sum((s_sim - s_obs)**2)

        if d < best_dist:
            best_dist = d
            best_alpha = alpha_candidate

    return best_alpha

Commentaires : 
Cette étape vise à mettre à jour le paramètre global $\alpha$, qui contrôle la distribution des variables latentes $\beta_j$ et donc la variabilité des paramètres $\theta_j$ entre les séries.
On construit une statistique résumée des paramètres actuels $\theta_j$ (ici leur moyenne), puis on génère $N_{\alpha}$ candidats $\alpha$ à partir d’une loi proposition. Pour chaque candidat, on simule de nouveaux $\beta_j \sim \text{Dirichlet}(\alpha)$, que l’on transforme en $\theta_j$, puis on calcule la même statistique résumée.
Le candidat $\alpha$ est sélectionné s’il permet de reproduire au mieux la structure observée des $\theta_j$. Cette étape correspond à une approximation ABC de la distribution conditionnelle $\pi(\alpha \mid \theta)$, en comparant des résumés globaux des paramètres.

## Mise à jour du paramètre $\sigma$

In [ ]:
def update_sigma_j(x_obs, theta, T, N_sigma=100):
    s_obs = summary_stats(x_obs)

    best_sigma = None
    best_dist = np.inf

    for _ in range(N_sigma):
        # prior Inverse-Gamma simulée via 1 / Gamma
        sigma2 = invgamma.rvs(1, 1)
        sigma = np.sqrt(sigma2)

        x_sim = simulate_ma2(theta, sigma, T)
        s_sim = summary_stats(x_sim)

        d = distance(s_sim, s_obs)

        if d < best_dist:
            best_dist = d
            best_sigma = sigma

    return best_sigma

Commentaires : 
Cette étape consiste à mettre à jour le paramètre de bruit $\sigma$ pour chaque série, conditionnellement aux paramètres $\theta$ courants.
On génère $N_{\sigma}$ candidats en échantillonnant $\sigma^2$ selon une loi Inverse-Gamma (via $1/\text{Gamma}$), puis on simule des séries temporelles à partir de ces valeurs. Pour chaque candidat, on calcule les statistiques résumées et on les compare à celles des données observées.
Le paramètre $\sigma$ retenu est celui qui minimise la distance entre les statistiques simulées et observées. Cette procédure correspond à une approximation ABC de la distribution conditionnelle $\pi(\sigma \mid \theta, x)$.
L’inclusion de $\sigma$ dans les mises à jour permet de prendre en compte l’incertitude sur le niveau de bruit du modèle, à condition que les statistiques résumées (notamment la variance) permettent de l’identifier.

## Algorithme Gibbs-ABC

In [ ]:
# Algorithme Gibbs-ABC
def gibbs_abc(x_data, n_iter=1000, N_theta=1000, N_alpha=100, N_sigma=100):
    n = len(x_data)
    T = len(x_data[0])

    # init
    thetas = [np.array([0.0, 0.0]) for _ in range(n)]
    alpha = np.array([1.0, 1.0, 1.0])
    sigmas = [1.0]*n

    samples_theta = []
    samples_alpha = []
    samples_sigma = []

    for it in range(n_iter):
        # on update ici theta_j
        for j in range(n):
            thetas[j] = update_theta_j(x_data[j], alpha, sigmas[j], T, N_theta)

        # on update alpha
        alpha = update_alpha(np.array(thetas), N_alpha)

        # --- update sigma_j ---
        for j in range(n):
            sigmas[j] = update_sigma_j(x_data[j], thetas[j], T, N_sigma)

        samples_theta.append(np.copy(thetas))
        samples_alpha.append(np.copy(alpha))
        samples_sigma.append(np.copy(sigmas))

    return samples_theta, samples_alpha, samples_sigma

Commentaires : 
Cette fonction implémente l’algorithme Gibbs-ABC pour estimer les paramètres du modèle hiérarchique MA(2).

Le principe repose sur une alternance de mises à jour conditionnelles des paramètres comme dans un échantillonneur de Gibbs mais en remplaçant les lois conditionnelles inaccessibles par des approximations ABC.

À chaque itération :
- on met à jour successivement les paramètres locaux $\theta_j$ pour chaque série, conditionnellement au paramètre global $\alpha$, en utilisant une étape ABC basée sur les statistiques résumées des données ;
- puis on met à jour le paramètre global $\alpha$ à partir des $\theta_j$ courants, également via une étape ABC.

Les valeurs des paramètres sont stockées à chaque itération afin d’approximer leur distribution a posteriori. Après une phase de burn-in, ces échantillons permettent d’estimer les paramètres et d’analyser leur incertitude.

Cette approche permet d’exploiter la structure hiérarchique du modèle tout en évitant le calcul explicite de la vraisemblance, et améliore l’efficacité par rapport à un ABC standard en décomposant le problème en mises à jour de plus faible dimension.

## Génération des données
Les données seront également utilisées dans la suite, afin de permettre la comparaison des algorithmes.

In [ ]:
# génération des données légerement différente --- soucis du sigma 2

def sample_mu(alpha, rng):
    """
    Tire mu depuis Dirichlet(alpha) puis applique la transformation
    beta -> mu qui garantit (mu1, mu2) dans la région de stationnarité M_2.
    """
    beta   = rng.dirichlet(alpha)
    b1, b2 = beta[0], beta[1]
    return np.array([b1 - b2, 2*(b1 + b2) - 1])


def sample_sigma2(zeta, rng):
    """
    Tire sigma² ~ IG(zeta[0], zeta[1]).
    Clip à 100 : des valeurs extrêmes de zeta[0] proches de 0
    peuvent donner sigma² >> 100. Avec les vrais paramètres,
    sigma² est de l'ordre de 1 à 5 — le clip ne biaise pas l'inférence.
    """
    sigma2 = invgamma.rvs(zeta[0], scale=zeta[1], random_state=rng)
    return min(float(sigma2), 100.0)


def sample_hyperparams(rng):
    """
    Tire les hyperparamètres depuis leurs priors (Section 10 du supplément) :
      alpha ~ Exp(1)^3
      zeta  ~ C+^2  (demi-Cauchy standard)
    zeta est simulée via tan(U) avec U ~ Unif(0, pi/2).
    Le clip à 100 dans sample_sigma2 gère les rares cas où zeta[0] ~ 0
    donne sigma² -> infini.
    """
    alpha_s = rng.exponential(scale=1.0, size=3)
    z1      = np.tan(rng.uniform(0, np.pi / 2))
    z2      = np.tan(rng.uniform(0, np.pi / 2))
    return alpha_s, np.array([z1, z2])


def simulate_ma2_sig(mu, sigma2, T, rng):
    """
    Simule une série MA(2) de longueur T.
    x(t) = y_{t+2} + mu[0]*y_{t+1} + mu[1]*y_t,  y_t ~ N(0, sigma2)
    On génère T+2 innovations pour avoir accès aux lags 1 et 2
    dès le premier pas de temps.
    """
    sig = np.sqrt(sigma2)
    y   = rng.normal(0, sig, T + 2)
    return np.array([y[t+2] + mu[0]*y[t+1] + mu[1]*y[t] for t in range(T)])


def generate_data(n, T, alpha, zeta, rng=None):
    """
    Génère n séries MA(2) hiérarchiques de longueur T.
    Retourne (x_list, mu_list, sigma2_list).
    """
    if rng is None:
        rng = np.random.default_rng(42)
    x_list, mu_list, sigma2_list = [], [], []
    for _ in range(n):
        mu     = sample_mu(alpha, rng)
        sigma2 = sample_sigma2(zeta, rng)
        x      = simulate_ma2_sig(mu, sigma2, T, rng)
        x_list.append(x)
        mu_list.append(mu)
        sigma2_list.append(sigma2)
    return x_list, mu_list, alpha, sigma2_list

# Pour générer les données
def generate_dataset(n_series=5, T=100):
    true_alpha = np.array([1.0, 2.0, 3.0])

    thetas_true = []
    x_data = []
    sigmas_true = []

    for j in range(n_series):
        beta = dirichlet.rvs(true_alpha)[0]
        theta = beta_to_theta(beta)

        sigma_true = np.sqrt(1 / np.random.gamma(2.0, 1.0))
        sigmas_true.append(sigma_true)
        x = simulate_ma2(theta, sigma=sigma_true, T=T)

        thetas_true.append(theta)
        x_data.append(x)

    return x_data, np.array(thetas_true), true_alpha, sigmas_true

Commentaires : 
Avant d’appliquer l’algorithme, on génère un jeu de données synthétique à partir du modèle hiérarchique.

On fixe un paramètre global $\alpha = (1,2,3)$ (donné par le papier) puis pour chaque série $j$ :
- on échantillonne $\beta_j \sim \text{Dirichlet}(\alpha)$,
- on transforme $\beta_j$ en paramètres $\theta_j$ du modèle MA(2),
- on simule une série temporelle à partir de ces paramètres. On simule 5 séries temporelles (donné par le papier). 

In [ ]:
# Générer des données
x_obs, mu_true, true_alpha, sigma2_true = generate_data(n, T, alpha, zeta, rng_data)

start = time.time()

# Lancer Gibbs-ABC
samples_theta, samples_alpha, samples_sigma = gibbs_abc(
    x_obs,
    n_iter=1000,
    N_theta=1000,
    N_alpha=100,
    N_sigma=100
)
end = time.time()
cpu_time = end - start

print("CPU time:", cpu_time)

In [ ]:
## Burn-in et estimation
burnin = 100

# Commentaires : 
# On enlève les premières itérations pour supprimer la phase transitoire.
# Ensuite on estime les paramètres via la moyenne empirique.

theta_samples = samples_theta[burnin:]
alpha_samples = samples_alpha[burnin:]

theta_est = np.mean(theta_samples, axis=0)
alpha_est = np.mean(alpha_samples, axis=0)

## Évaluation des paramètres

On compare les estimations aux vraies valeurs.

In [ ]:
print("True theta (first series):", mu_true[0])
print("Estimated theta:", theta_est[0])

Commentaires du résultat : $\theta_1$ est bien estimé : l’écart est faible donc on a une bonne récupération de la structure principale du MA(2). $\theta_2$ est plus biaisé : surestimation visible (0.11 au lieu de 0.199), ce qui est classique en ABC avec statistiques résumées imparfaites. Ce type d’écart vient principalement de la perte d’information dans les statistiques résumées (ACF / corrélations), de l'identifiabilité partielle du paramètre $\theta_2$ dans un MA(2) et de l'approximation ABC (on ne travaille pas avec la vraisemblance exacte). 

## Visualisation : Posterior vs Prior

In [ ]:
# Échantillon prior pour comparaison visuelle
rng_pv    = np.random.default_rng(SEED + 99)
mu1_prior = []
for _ in range(5000):
    a_i, z_i = sample_hyperparams(rng_pv)
    mu1_prior.append(sample_mu(a_i, rng_pv)[0])
mu1_prior = np.array(mu1_prior)

# Conversion en array
theta_samples = np.array(samples_theta[burnin:])

n_series = theta_samples.shape[1]

prior_theta1 = np.array(mu1_prior)

# --- Plot pour les 5 séries ---
fig, axes = plt.subplots(1, n_series, figsize=(20,4))

for j in range(n_series):
    
    # Posterior θ1 pour série j
    theta1_post = theta_samples[:, j, 0]
    
    axes[j].hist(theta1_post, bins=30, density=True, alpha=0.6, label="Posterior")
    axes[j].hist(prior_theta1, bins=30, density=True, alpha=0.4, label="Prior")
    
    # vraie valeur
    axes[j].axvline(mu_true[j][0], linestyle='--', label="True value")
    
    axes[j].set_title(f"Series {j+1}")
    
# légende globale
axes[0].legend()

plt.suptitle("Posterior vs Prior for θ1 (all series)")
plt.savefig("Results/Gibbs_ABC_posterior_vs_prior.png", dpi=150, bbox_inches="tight")
plt.show()


Commentaires : 
Ce code permet de comparer, pour chaque série, la distribution a posteriori du paramètre $\theta_1$ (issue de l’algorithme Gibbs-ABC) avec sa distribution a priori.

Les échantillons posterior sont obtenus après burn-in à partir de la chaîne MCMC. La prior est générée en échantillonnant $\beta \sim \text{Dirichlet}(\alpha)$, puis en transformant en $\theta$. Pour chaque série, on trace un histogramme du posterior et de la prior, ainsi qu’une ligne verticale indiquant la vraie valeur du paramètre.

Interprétation : 

La prior est généralement large : elle représente l’incertitude avant observation des données. Le posterior est plus concentré : il reflète l’information apportée par les données. Si l’algorithme fonctionne bien, le posterior est centré autour de la vraie valeur (ligne pointillée). Cela est particulièrement vrai pour les séries 1, 4 et 5. 

Ainsi, ces histogrammes permettent de vérifier visuellement que l’algorithme apprend correctement les paramètres à partir des données.

PS : on ne visualise que theta 1 comme dans le papier pour des soucis de simplification de la présentation des résultats. 

## Comparaison des algorithmes : métriques d’évaluation

Commentaires : Afin de comparer les performances de l’algorithme Gibbs-ABC et de l’ABC par rejet, on utilise trois métriques complémentaires : le temps de calcul (CPU time), l’erreur d’inférence et l’erreur de Monte Carlo.

Le **CPU time** mesure le temps d’exécution de l’algorithme. Il permet d’évaluer le coût computationnel et l’efficacité pratique de chaque méthode. Un algorithme plus rapide est préférable à précision équivalente, mais un compromis est souvent nécessaire entre vitesse et qualité des estimations.

L’**erreur d’inférence** mesure la distance entre les paramètres estimés et les vraies valeurs utilisées pour générer les données. Elle reflète la capacité de l’algorithme à reconstruire correctement les paramètres du modèle. Une erreur faible indique une bonne précision statistique.

L’**erreur de Monte Carlo** correspond à la variabilité des échantillons générés par l’algorithme, généralement mesurée via leur variance. Elle renseigne sur la stabilité et la qualité de l’exploration de la distribution a posteriori. Une faible erreur de Monte Carlo indique une chaîne stable et bien mélangée.

Ces trois métriques permettent ainsi une évaluation complète : le CPU time capture le coût computationnel, l’erreur d’inférence mesure la précision des estimations, et l’erreur de Monte Carlo évalue la fiabilité et la robustesse de l’algorithme. Ensemble, elles offrent une base pertinente pour comparer Gibbs-ABC et ABC-reject.

### Temps de calcul - CPU Time

In [ ]:
print("CPU time:", cpu_time)

### Erreur d'inférence

In [ ]:
theta_est = np.mean(samples_theta[burnin:], axis=0)

# erreur sur toutes les séries
errors = [np.linalg.norm(theta_est[j] - mu_true[j]) for j in range(len(mu_true))]

inferential_error = np.mean(errors)

print("Inferential error:", inferential_error)

### Erreur Monte Carlo

In [ ]:
theta_samples = np.array(samples_theta[burnin:])

# variance moyenne sur les paramètres
mc_error = np.mean(np.var(theta_samples, axis=0))

print("Monte Carlo error:", mc_error)

Commentaires : 
Le temps de calcul est relativement élevé (environ 29 minutes), ce qui est attendu pour un algorithme de type Gibbs-ABC hiérarchique. L’approche est coûteuse en temps de calcul, ce qui est un compromis classique des méthodes ABC, mais reste acceptable pour une étude expérimentale.
Concernant l'erreur d'inférence, on voit que le modèle parvient à capturer la structure globale des paramètres mais avec une légère déviation due à l’approximation ABC et à l’utilisation de statistiques résumées imparfaites.
Enfin, l'erreur Monte Carlo est faible. L’algorithme est numériquement stable : les fluctuations dues au hasard de simulation sont faibles, ce qui indique une bonne convergence de la chaîne.

# Partie II : ABC Reject (Vanilla)


## ABC : le principe

Puisqu'on ne peut pas évaluer la vraisemblance, on contourne le problème :

1. Tirer $\theta \sim \pi(\theta)$ depuis le prior
2. Simuler $x^{\text{sim}} \sim f(\cdot \mid \theta)$
3. Comparer des statistiques résumées $s(x^{\text{sim}})$ et $s(x^\star)$
4. Accepter $\theta$ si $\delta(s(x^{\text{sim}}), s(x^\star)) < \varepsilon$

### Pourquoi ne pas comparer les séries directement

Si tu compares $x^{\text{sim}}$ et $x^\star$ avec la distance euclidienne dans $\mathbb{R}^{100}$, tu obtiens une distance qui ne dépend presque pas de $\theta$. Voici pourquoi.

Même avec le bon $\theta^\star$, les deux séries utilisent des innovations $y_t$ différentes et indépendantes. On peut montrer que :

$$\mathbb{E}\left[\|x^{\text{sim}} - x^\star\|_2^2\right] = 2T\sigma^2(1 + \mu_1^2 + \mu_2^2)$$

Ce terme croît avec $T$, indépendamment de la qualité de $\theta$. Pour $T=100$, la distance euclidienne entre deux séries avec le bon paramètre est déjà de l'ordre de $\sqrt{200} \approx 14$. Elle est quasi-identique avec un mauvais paramètre. La distance brute ne t'informe pas sur $\theta$.

### La malédiction de la dimension

Le taux d'acceptation d'ABC décroît exponentiellement avec la dimension $d$ des statistiques :

$$P(\delta < \varepsilon) \approx \varepsilon^d$$

- $d=2$, $\varepsilon=0.1$ : probabilité $\approx 0.01$ — acceptable
- $d=100$ (données brutes) : probabilité $\approx 10^{-100}$ — impossible

---

## 2. Principe ABC-Reject et malédiction de la dimension

### Algorithme 1 — Vanilla ABC

> Pour $i = 1, \ldots, N_\text{tot}$, répéter :
> 1. Tirer $\theta^{(i)} \sim \pi(\theta)$
> 2. Simuler $x^{(i)} \sim f(\cdot \mid \theta^{(i)})$
> 3. Accepter si $\delta(x^{(i)}, x^\star) < \varepsilon$
>
> Retourner les $N$ paramètres avec les plus petites distances.

Le posterior approché est $\pi_\varepsilon\{\theta \mid s(x^\star)\} \propto \int \pi(\theta) f(x \mid \theta) \mathbf{1}_{d < \varepsilon} \mathrm{d}x$.

### Malédiction de la dimension

Le taux d'acceptation s'effondre exponentiellement : $P(d < \varepsilon) \approx \varepsilon^d$.

| Dimension $d$ | $\varepsilon = 0.1$ | $\varepsilon = 0.5$ |
|:---:|:---:|:---:|
| 2 | $1\%$ | $25\%$ |
| 10 | $10^{-10}$ | $\sim 10^{-3}$ |
| 20 | $10^{-20}$ | $\sim 10^{-6}$ |

Avec $d = 20$, le posterior reste quasi-identique au prior — ABC-Reject n'apprend presque rien.


## Statistiques résumées

### Statistique suffisante, quasi-suffisante, injective : ce que ça veut dire

Une statistique $s(x)$ est **suffisante** pour $\theta$ si elle capture toute l'information des données sur $\theta$. Formellement : la distribution de $x$ conditionnellement à $s(x)$ ne dépend pas de $\theta$. Connaître $s(x)$ est aussi informatif que connaître $x$ entier.

En pratique, une statistique parfaitement suffisante de petite dimension n'existe généralement pas. On parle de statistique **quasi-suffisante** quand elle capture l'essentiel de l'information avec une perte faible. Un résultat théorique de Fearnhead et Prangle (2012) dit que la dimension de $s$ doit être égale à la dimension de $\theta$ — ni plus (le taux d'acceptation s'effondre), ni moins (on perd de l'information).

Une statistique est **injective** si $s(\theta_1) = s(\theta_2) \implies \theta_1 = \theta_2$. Autrement dit : deux paramètres différents donnent forcément deux valeurs de statistique différentes. Sans injectivité, ABC ne peut pas distinguer deux paramètres différents — l'inférence est impossible.

Pour le MA(2), les autocorrélations théoriques valent :

$$\rho_1(\mu) = \frac{\mu_1(1+\mu_2)}{1+\mu_1^2+\mu_2^2}, \qquad \rho_2(\mu) = \frac{\mu_2}{1+\mu_1^2+\mu_2^2}$$

La fonction $\mu \mapsto (\rho_1(\mu), \rho_2(\mu))$ est injective sur $\mathcal{M}_2$ : deux paramètres $\mu \neq \mu'$ donnent toujours $(\rho_1, \rho_2) \neq (\rho_1', \rho_2')$. Les autocorrélations empiriques $(\hat{\rho}_1, \hat{\rho}_2)$ forment donc une statistique quasi-suffisante pour $\mu$.

### Distance $w(x_j)$ pour $\mu_j$

$$w(x_j) = \sqrt{(\hat{\rho}_1(x_j^{\text{sim}}) - \hat{\rho}_1(x_j^\star))^2 + (\hat{\rho}_2(x_j^{\text{sim}}) - \hat{\rho}_2(x_j^\star))^2}$$

L'autocorrélation empirique au lag $k$ :
$$\hat{\rho}_k = \frac{\sum_{t=1}^{T-k}(x_t - \bar{x})(x_{t+k} - \bar{x})}{\sum_{t=1}^{T}(x_t - \bar{x})^2}$$

Le `+ 1e-12` au dénominateur évite une division par zéro si la série est constante. Cela arrive lors de simulations pathologiques depuis le prior (sigma² très grand). On retourne alors 0 au lieu de `NaN`.

### Distance $v(x_j)$ pour $\sigma_j^2$

$$v(x_j) = \left|\frac{1}{M}\sum_{t=1}^{M}(x_j(3t) - \bar{x}_j^{\text{thin}})^2 - \frac{1}{M}\sum_{t=1}^{M}(x_j^\star(3t) - \bar{x}_j^{\star\,\text{thin}})^2\right|$$

avec $M = \lfloor T/3 \rfloor$. Les observations $x(3), x(6), x(9), \ldots$ sont indépendantes (propriété MA(2) rappelée en introduction), donc elles forment un échantillon iid de variance $\sigma^2(1+\mu_1^2+\mu_2^2)$. Cette variance est quasi-suffisante pour $\sigma^2$.

### Distance globale $\delta(x)$ et calibration des quantiles

$$\delta(x) = \sum_{j=1}^{n} \left(\frac{w(x_j)}{q_j} + \frac{v(x_j)}{q'_j}\right)$$

Sans les dénominateurs $q_j$ et $q'_j$, $w$ et $v$ ne sont pas comparables :
- $w$ est une distance entre autocorrélations dans $[-1,1]$ : valeurs typiques entre 0.1 et 0.5
- $v$ est une différence de variances : valeurs typiques 10 à 100 fois plus grandes

Sans normalisation, $\delta$ est entièrement dominé par $v$. Les autocorrélations n'influencent plus les acceptations. On estime donc $q_j = Q_{0.1\%}(w_j)$ et $q'_j = Q_{0.1\%}(v_j)$ sous le prior. Après normalisation, $w/q_j$ et $v/q'_j$ ont des distributions similaires et contribuent à parts égales.

On choisit le quantile 0.1% plutôt que la moyenne ou l'écart-type parce que les distributions de $w$ et $v$ ont des queues lourdes — leur moyenne et écart-type sont instables. Le quantile 0.1% est robuste et capture l'échelle des petites distances qui nous intéressent pour l'acceptation.

In [ ]:
def autocorr_emp(x, lag):
    """
    Autocorrélation empirique au lag donné.
    Mesure la corrélation linéaire entre x(t) et x(t+lag).
    Le +1e-12 au dénominateur évite une division par zéro si la
    série est constante (simulation pathologique depuis le prior).
    """
    xc = x - np.mean(x)
    return np.sum(xc[:len(x)-lag] * xc[lag:]) / (np.sum(xc**2) + 1e-12)


def compute_w(x_sim, x_obs_j):
    """
    Distance w entre les autocorrélations lags 1 et 2 de x_sim et x_obs_j.
    Quasi-suffisante pour mu_j : rho1 et rho2 sont des fonctions injectives
    de (mu1, mu2) sur M_2, donc w=0 implique mu_sim = mu_obs.
    """
    d1 = autocorr_emp(x_sim, 1) - autocorr_emp(x_obs_j, 1)
    d2 = autocorr_emp(x_sim, 2) - autocorr_emp(x_obs_j, 2)
    return np.sqrt(d1**2 + d2**2)


def compute_v(x_sim, x_obs_j):
    """
    Distance v entre les variances des observations espacées de 3 pas.
    x(t) et x(t+3) sont indépendants dans MA(2) : x(3), x(6), ...
    forment un échantillon iid de variance sigma²(1+mu1²+mu2²).
    Quasi-suffisante pour sigma².
    """
    M   = T // 3
    idx = np.arange(1, M + 1) * 3 - 1
    return abs(np.var(x_sim[idx]) - np.var(x_obs_j[idx]))


def delta_global(x_sim_list, x_obs_list, q_w, q_v):
    """
    Distance globale delta(x) normalisée — Équation (4) du papier.
    Somme sur les n séries de w/q_w + v/q_v.
    Retourne np.inf si une valeur n'est pas finie (simulation rejetée).
    """
    total = 0.0
    for j in range(n):
        w_j = compute_w(x_sim_list[j], x_obs_list[j])
        v_j = compute_v(x_sim_list[j], x_obs_list[j])
        if not np.isfinite(w_j) or not np.isfinite(v_j):
            return np.inf
        total += w_j / (q_w[j] + 1e-12) + v_j / (q_v[j] + 1e-12)
    return total

## Données observées $x^\star$

In [ ]:
def rho1_theorique(mu):
    return mu[0] * (1 + mu[1]) / (1 + mu[0]**2 + mu[1]**2)

def rho2_theorique(mu):
    return mu[1] / (1 + mu[0]**2 + mu[1]**2)

def dans_M2(mu):
    m1, m2 = mu[0], mu[1]
    return abs(m2) < 1 and m2 + m1 > -1 and m2 - m1 > -1

print("Toy dataset — vrais parametres (Section 10.2)")
print("=" * 70)
print(f"{'Serie':<7} {'mu1':>7} {'mu2':>7} {'sigma2':>8}  "
      f"{'rho1_th':>9} {'rho1_emp':>9} {'rho2_th':>9} {'rho2_emp':>9}  {'M2?':>5}")
print("-" * 70)
for j in range(n):
    mu_j = np.array(mu_true[j])
    s2   = sigma2_true[j]
    r1t  = rho1_theorique(mu_j)
    r2t  = rho2_theorique(mu_j)
    r1e  = autocorr(x_obs[j], 1)
    r2e  = autocorr(x_obs[j], 2)
    ok   = "OK" if dans_M2(mu_j) else "WARN"
    warn = " <-- sigma2 eleve!" if s2 > 10 else ""
    print(f"  {j+1:<5} {mu_j[0]:>7.3f} {mu_j[1]:>7.3f} {s2:>8.3f}  "
          f"{r1t:>9.3f} {r1e:>9.3f} {r2t:>9.3f} {r2e:>9.3f}  {ok:>5}{warn}")
print()
print("rho_th = valeur theorique exacte  |  rho_emp = valeur empirique sur T=100")
print("Ecart typique : 1/sqrt(T) ~ 0.1  |  M2? : stationnarite garantie par construction")

fig, axes = plt.subplots(n, 1, figsize=(12, 8), sharex=True)
for j in range(n):
    axes[j].plot(x_obs[j], color=f'C{j}', linewidth=0.8)
    axes[j].axhline(0, color='gray', linewidth=0.4, linestyle='--')
    axes[j].set_ylabel(f'$x_{j+1}$', fontsize=9)
    axes[j].set_title(
        rf'$\mu^\star={np.array(mu_true[j]).round(3)}$, '
        rf'$\sigma^2={sigma2_true[j]:.2f}$, '
        rf'$\hat\rho_1={autocorr(x_obs[j],1):.3f}$, '
        rf'$\hat\rho_2={autocorr(x_obs[j],2):.3f}$',
        fontsize=8, loc='left')
axes[-1].set_xlabel('Temps $t$', fontsize=10)
fig.suptitle(r"Donnees observees $x^\star$ — $n=5$ series MA(2), $T=100$",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('Results/fig_donnees_observees.png', dpi=150, bbox_inches='tight')
plt.show()

## Calibration des quantiles $q_j$ et $q'_j$

On simule $N_{\text{cal}} = 10\,000$ jeux de paramètres depuis le prior complet pour estimer les distributions de $w$ et $v$. On prend ensuite le quantile 0.1% de chaque distribution comme normalisateur. On utilise `nanquantile` pour ignorer les rares `NaN` sans interrompre le calcul.

In [ ]:
N_CAL   = 100_000
rng_cal = np.random.default_rng(SEED)

print(f"Calibration sur {N_CAL:,} simulations depuis le prior hierarchique...")
t0 = time.perf_counter()

w_cal = np.zeros((N_CAL, n))
v_cal = np.zeros((N_CAL, n))

for i in range(N_CAL):
    alpha_i, zeta_i = sample_hyperparams(rng_cal)
    x_sim_i, _, _,_   = generate_data(n, T, alpha_i, zeta_i, rng_cal)
    for j in range(n):
        w_cal[i, j] = compute_w(x_sim_i[j], x_obs[j])
        v_cal[i, j] = compute_v(x_sim_i[j], x_obs[j])

print(f"  Termine en {time.perf_counter()-t0:.1f}s")

q_w = np.nanquantile(w_cal, 0.001, axis=0)
q_v = np.nanquantile(v_cal, 0.001, axis=0)

print("\nQuantiles Q(0.1%) :")
print(f"{'Serie':<8} {'q_w':>12} {'q_v':>12}")
print("-" * 34)
for j in range(n):
    print(f"  {j+1:<6} {q_w[j]:>12.5f} {q_v[j]:>12.5f}")

# Verification : delta avec les vrais params doit etre ~ 400-600 (coherent avec le papier)
rng_chk = np.random.default_rng(SEED + 99)
d_check = []
for _ in range(50):
    xs, _, _,_ = generate_data(n, T, alpha, zeta, rng_chk)
    d_check.append(delta_global(xs, x_obs, q_w, q_v))
print(f"\nVerification normalisation — delta avec vrais params :")
print(f"  mean={np.mean(d_check):.1f}, std={np.std(d_check):.1f}")
print(f"  (attendu ~ 400-600 pour etre coherent avec la PPD du papier = 436.8)")

fig, axes = plt.subplots(2, n, figsize=(14, 5))
for j in range(n):
    for row, (data, qval, color, ylabel) in enumerate([
        (w_cal[:, j], q_w[j], 'steelblue',  'w(x_j)'),
        (v_cal[:, j], q_v[j], 'darkorange', 'v(x_j)')
    ]):
        ax = axes[row, j]
        finite = data[np.isfinite(data)]
        if row == 1:
            finite = finite[finite <= np.percentile(finite, 95)]
        ax.hist(finite, bins=50, density=True, color=color, alpha=0.7)
        ax.axvline(qval, color='red', lw=2, linestyle='--',
                   label=f'Q 0.1% = {qval:.4f}')
        ax.set_title(f'Serie {j+1}', fontsize=9)
        ax.legend(fontsize=7)
        if j == 0:
            ax.set_ylabel(ylabel, fontsize=9)
fig.suptitle(f'Calibration — distributions de w et v sous le prior (N_CAL={N_CAL:,})',
             fontsize=11)
plt.tight_layout()
plt.savefig('Results/fig_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## ABC-Reject

In [ ]:
def abc_reject(x_obs, q_w, q_v, N_tot, N_keep, seed,
               verbose=True, print_every=100_000):
    # ABC-Reject (Algorithme 1 du papier) sur le modele MA(2) hierarchique.
    # Stocke toutes les simulations, retourne les N_keep avec les plus petites distances.
    rng         = np.random.default_rng(seed)
    alphas_all  = np.zeros((N_tot, 3))
    zetas_all   = np.zeros((N_tot, 2))
    mus_all     = np.zeros((N_tot, n, 2))
    sigma2s_all = np.zeros((N_tot, n))
    dists_all   = np.full(N_tot, np.inf)

    t0 = time.perf_counter()

    for i in range(N_tot):
        alpha_i, zeta_i         = sample_hyperparams(rng)
        x_sim_i, mu_i,_, sigma2_i = generate_data(n, T, alpha_i, zeta_i, rng)
        dists_all[i]   = delta_global(x_sim_i, x_obs, q_w, q_v)
        alphas_all[i]  = alpha_i
        zetas_all[i]   = zeta_i
        mus_all[i]     = np.array(mu_i)
        sigma2s_all[i] = np.array(sigma2_i)

        if verbose and (i + 1) % print_every == 0:
            el    = time.perf_counter() - t0
            speed = (i + 1) / el
            ok    = np.isfinite(dists_all[:i+1])
            eps   = (np.sort(dists_all[:i+1][ok])[min(N_keep-1, ok.sum()-1)]
                     if ok.sum() >= N_keep else float('nan'))
            print(f"  [{i+1:>9,}/{N_tot:,}]  eps={eps:6.2f}  "
                  f"{speed:5.0f} sim/s  ETA={int((N_tot-i-1)/speed)}s")

    cpu = time.perf_counter() - t0
    idx = np.argsort(dists_all)[:N_keep]
    eps = float(dists_all[idx[-1]])

    print(f"\nSimulations valides : {np.sum(np.isfinite(dists_all)):,}/{N_tot:,}")
    print(f"Seuil effectif eps  : {eps:.4f}")
    print(f"Taux d'acceptation  : {N_keep/N_tot:.4%}")
    print(f"Temps CPU           : {cpu:.1f}s  ({N_tot/cpu:.0f} sim/s)")

    return dict(
        alphas          = alphas_all[idx],
        zetas           = zetas_all[idx],
        mus             = mus_all[idx],
        sigma2s         = sigma2s_all[idx],
        distances       = dists_all[idx],
        all_distances   = dists_all,
        epsilon         = eps,
        cpu_time        = cpu,
        n_simulations   = N_tot,
        acceptance_rate = N_keep / N_tot,
    )

In [ ]:
# Lancement — avec N_TOT=1_100_000 comptez 15min environ sur un CPU moderne.
# Decommenter N_TOT=50_000 dans la cellule parametres pour un test rapide
res = abc_reject(x_obs, q_w, q_v, N_TOT, N_KEEP, seed=SEED)

## Résultats et figures

### Vérification numérique

In [ ]:
print(f"NaN dans distances  : {np.sum(np.isnan(res['distances']))}")
print(f"Range des distances : [{res['distances'].min():.2f}, {res['distances'].max():.2f}]")
print(f"Seuil epsilon       : {res['epsilon']:.4f}")
print(f"Taux d'acceptation  : {res['acceptance_rate']:.4%}")

## Résultats : posterior vs prior -- Figure 11 du papier

Si posterior $\approx$ prior, ABC-Reject n'a rien appris (malédiction de la dimension, $d=20$).

In [ ]:
print(f"NaN dans distances  : {np.sum(np.isnan(res['distances']))}")
print(f"Range des distances : [{res['distances'].min():.2f}, {res['distances'].max():.2f}]")

In [ ]:
rng_pv    = np.random.default_rng(SEED + 99)
mu1_prior = []
for _ in range(5000):
    a_i, z_i = sample_hyperparams(rng_pv)
    mu1_prior.append(sample_mu(a_i, rng_pv)[0])
mu1_prior = np.array(mu1_prior)

mu1_post = res['mus'][:, 0, 0]
mu1_true = float(mu_true[0][0])
xr       = np.linspace(-1.1, 1.1, 300)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mu1_post, bins=40, density=True, color='steelblue', alpha=0.75,
        label='Posterior ABC-Reject')
ax.plot(xr, gaussian_kde(mu1_prior)(xr), 'k-', lw=1.8, label='Prior')
ax.axvline(mu1_true, color='red', lw=2.5, label=f'Vraie valeur = {mu1_true:.3f}')
ax.set_xlabel(r'$\mu_{1,1}$', fontsize=13)
ax.set_ylabel('Densite', fontsize=11)
ax.set_title('Posterior de $\\mu_{1,1}$ — ABC-Reject vs prior\n'
             '(Reproduction Figure 11 du papier, Section 10.2)', fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(-1.1, 1.1)
plt.tight_layout()
plt.savefig('Results/fig_posterior_mu1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Vraie valeur  mu1*   : {mu1_true:.4f}")
print(f"Moyenne posterior    : {np.mean(mu1_post):.4f}")
print(f"Biais                : {np.mean(mu1_post) - mu1_true:+.4f}")
print(f"Ecart-type posterior : {np.std(mu1_post):.4f}  vs  prior : {np.std(mu1_prior):.4f}")
print()
print("Conclusion : posterior ≈ prior -> ABC-Reject n'apprend presque rien.")
print("C'est la signature de la malediction de la dimension (d=20).")

### Toutes les marginales $\mu_{j,1}$

In [ ]:
prior_kde = gaussian_kde(mu1_prior)
fig, axes = plt.subplots(1, n, figsize=(15, 3.5), sharey=True)
for j in range(n):
    ax      = axes[j]
    mu_post = res['mus'][:, j, 0]
    mu_ref  = float(mu_true[j][0])
    ax.hist(mu_post, bins=35, density=True, color='steelblue', alpha=0.75, label='ABC-Reject')
    ax.plot(xr, prior_kde(xr), 'k-', lw=1.3, alpha=0.75, label='Prior')
    ax.axvline(mu_ref, color='red', lw=2, label=f'vrai = {mu_ref:.2f}')
    ax.set_title(f'Serie {j+1}', fontsize=10)
    ax.set_xlabel(r'$\mu_{j,1}$', fontsize=10)
    ax.legend(fontsize=7)
    if j == 0: ax.set_ylabel('Densite', fontsize=10)
fig.suptitle('ABC-Reject — Posteriors marginaux de $\\mu_{j,1}$ pour les 5 series\n'
             'Dans tous les cas : posterior ≈ prior', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('Results/fig_posterior_toutes_series.png', dpi=150, bbox_inches='tight')
plt.show()

### Analyse des distances

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1 : distribution des distances acceptees
ax = axes[0]
ax.hist(res['distances'], bins=50, color='steelblue', alpha=0.8, density=True)
ax.axvline(res['epsilon'], color='red', linestyle='--', lw=2,
           label=f"eps = {res['epsilon']:.1f}")
ax.axvline(np.mean(res['distances']), color='orange', lw=1.5,
           label=f"Moy = {np.mean(res['distances']):.1f}")
ax.set_xlabel(r'Distance $\delta$', fontsize=11)
ax.set_title('Distances des N acceptes', fontsize=10)
ax.legend(fontsize=8)

# Panel 2 : acceptes vs prior
ax = axes[1]
rng_comp = np.random.default_rng(SEED + 500)
d_prior_sample = []
for _ in range(2000):
    a, z = sample_hyperparams(rng_comp)
    xs, _, _,_ = generate_data(n, T, a, z, rng_comp)
    d = delta_global(xs, x_obs, q_w, q_v)
    if np.isfinite(d): d_prior_sample.append(d)
d_prior_sample = np.array(d_prior_sample)

clip = np.percentile(d_prior_sample, 90)
ax.hist(d_prior_sample[d_prior_sample <= clip], bins=50, density=True,
        color='gray', alpha=0.5, label='Prior (2000 tirages)')
ax.hist(res['distances'][res['distances'] <= clip], bins=50, density=True,
        color='steelblue', alpha=0.7, label='ABC-Reject acceptes')
ax.axvline(np.mean(d_prior_sample), color='gray', lw=2, linestyle='--',
           label=f"Moy. prior = {np.mean(d_prior_sample):.0f}")
ax.axvline(np.mean(res['distances']), color='steelblue', lw=2,
           label=f"Moy. acc. = {np.mean(res['distances']):.0f}")
ax.set_xlabel(r'Distance $\delta$', fontsize=11)
ax.set_title('Acceptes vs prior', fontsize=10)
ax.legend(fontsize=7)

# Panel 3 : convergence de epsilon
ax = axes[2]
all_d = res['all_distances']
checkpoints = np.linspace(N_KEEP, len(all_d), 50, dtype=int)
epsilons = []
for k in checkpoints:
    ok_k = np.isfinite(all_d[:k])
    if ok_k.sum() >= N_KEEP:
        epsilons.append(np.sort(all_d[:k][ok_k])[N_KEEP-1])
    else:
        epsilons.append(np.nan)
ax.plot(checkpoints, epsilons, color='steelblue', lw=2)
ax.axhline(res['epsilon'], color='red', lw=1.5, linestyle='--',
           label=f"eps final = {res['epsilon']:.1f}")
ax.set_xlabel('Nombre de simulations', fontsize=11)
ax.set_ylabel(r'Seuil $\varepsilon$ courant', fontsize=11)
ax.set_title(r'Convergence du seuil $\varepsilon$', fontsize=10)
ax.legend(fontsize=8)

plt.suptitle('Analyse des distances — ABC-Reject sur MA(2) hierarchique',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('Results/fig_distances_analyse.png', dpi=150, bbox_inches='tight')
plt.show()

### Distance posterior prédictive — métrique principale

**Point clé :** on simule depuis $(\mu^{(i)}, \sigma^{2(i)})$ **directement** (pas depuis $(\alpha^{(i)}, \varsigma^{(i)})$).
Appeler `generate_data(alpha, zeta)` retirerait de nouveaux $\mu_j$ aléatoires, perdant l'information sélectionnée par ABC.


In [ ]:
def posterior_predictive_dist(res, x_obs, q_w, q_v, seed, n_eval=200):
    rng = np.random.default_rng(seed)
    idx = np.random.default_rng(seed+1).choice(len(res['mus']), n_eval, replace=False)
    dists = []
    for k in idx:
        tot = 0.0; ok = True
        for j in range(n):
            mu_ij     = res['mus'][k, j]
            sigma2_ij = res['sigma2s'][k, j]
            x_pred_j  = simulate_ma2(mu_ij, sigma2_ij, T, rng)
            w_j = compute_w(x_pred_j, x_obs[j])
            v_j = compute_v(x_pred_j, x_obs[j])
            if not (np.isfinite(w_j) and np.isfinite(v_j)):
                ok = False; break
            tot += w_j / (q_w[j] + 1e-12) + v_j / (q_v[j] + 1e-12)
        if ok and np.isfinite(tot): dists.append(tot)
    dists = np.array(dists)
    mean = float(np.mean(dists))
    sem  = float(np.std(dists) / np.sqrt(len(dists)))   # erreur standard
    return mean, sem, dists


def prior_predictive_dist(n_samples, x_obs, q_w, q_v, seed):
    # Baseline : prior predictive (aucun apprentissage).
    rng = np.random.default_rng(seed)
    dists = []
    for _ in range(n_samples):
        a, z = sample_hyperparams(rng)
        tot  = 0.0; ok = True
        for j in range(n):
            mu_j     = sample_mu(a, rng)
            sigma2_j = sample_sigma2(z, rng)
            x_pred_j = simulate_ma2(mu_j, sigma2_j, T, rng)
            w_j = compute_w(x_pred_j, x_obs[j])
            v_j = compute_v(x_pred_j, x_obs[j])
            if not (np.isfinite(w_j) and np.isfinite(v_j)):
                ok = False; break
            tot += w_j / (q_w[j] + 1e-12) + v_j / (q_v[j] + 1e-12)
        if ok and np.isfinite(tot): dists.append(tot)
    dists = np.array(dists)
    return float(np.mean(dists)), float(np.std(dists) / np.sqrt(len(dists)))


print("Calcul des distances predictives...")
mean_abc,   sem_abc,   dists_ppd = posterior_predictive_dist(
    res, x_obs, q_w, q_v, seed=SEED+100, n_eval=500)
mean_prior, sem_prior = prior_predictive_dist(500, x_obs, q_w, q_v, seed=SEED+200)

# Note methodologique
print()
print("Note : le +/- rapporte l'erreur standard de la moyenne (std / sqrt(n_eval)),")
print("comparable au +/- du papier qui est la std sur 100 replicats independants.")
print("La std brute des distances individuelles est bien plus grande (variabilite intra-echantillon).")
print()
print(f"{'─'*56}")
print(f"  Prior (baseline)        : {mean_prior:.1f} +/- {sem_prior:.1f}")
print(f"  ABC-Reject (notre impl) : {mean_abc:.1f} +/- {sem_abc:.1f}")
print(f"  ABC-Reject (papier)     : 436.8 +/- 1.6")
print(f"  ABC-Gibbs  (papier)     : 274.1 +/- 2.5  <- reference")
print(f"{'─'*56}")
print(f"  Ratio Reject / Gibbs    : {436.8/274.1:.2f}x  (Gibbs est meilleur)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dists_ppd, bins=35, density=True, color='steelblue', alpha=0.75,
        label=f'ABC-Reject : {mean_abc:.0f} +/- {sem_abc:.1f}')
ax.axvline(mean_abc,   color='steelblue',  lw=2.5)
ax.axvline(mean_prior, color='gray',       lw=1.5, linestyle=':',
           label=f'Prior (baseline) : {mean_prior:.0f}')
ax.axvline(274.1, color='darkorange', lw=2.5, linestyle='--',
           label='ABC-Gibbs (papier) : 274.1')
ax.set_xlabel(r'Distance predictive $\delta(\tilde{x}, x^\star)$', fontsize=11)
ax.set_ylabel('Densite', fontsize=11)
ax.set_title('Distance posterior predictive\n'
             'Plus petite = meilleur  |  +/- = erreur standard de la moyenne', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('Results/fig_predictive_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparaison des algorithmes : métriques d'évaluation

Trois métriques complémentaires permettent une évaluation complète :

| Métrique | Définition | Interprétation |
|----------|-----------|----------------|
| **CPU time** | Temps d'exécution total | Coût computationnel — plus rapide = meilleur à précision égale |
| **Erreur d'inférence** | $\|\overline\mu_j^{\text{ABC}} - \mu_j^\star\|_2$ moyenné sur $j$ | Précision statistique — proche de 0 = posterior centré sur la vraie valeur |
| **Erreur de Monte Carlo** | $\text{Var}(\mu_j^{(i)})$ moyennée sur $j$ | Stabilité de l'exploration — faible = chaîne bien mélangée |

Pour ABC-Reject (iid), l'erreur MC reflète directement la dispersion des $N$ échantillons acceptés.


In [ ]:
# ── 1. CPU TIME ──────────────────────────────────────────────────────────────
cpu_time = res['cpu_time']

# ── 2. ERREUR D'INFERENCE ────────────────────────────────────────────────────
# Inferential error : ecart entre pi_epsilon (cible ABC) et le vrai posterior.
# On l'approxime par la distance entre la moyenne posterior ABC et la vraie valeur,
# NORMALISEE par l'ecart-type prior pour avoir une mesure relative.
# Un ratio proche de 1 signifie que pi_epsilon ≈ prior -> ABC n'apprend rien.
# Un ratio proche de 0 signifie que pi_epsilon ≈ vrai posterior.
inferential_errors = []
for j in range(n):
    mu_j_post_mean = res['mus'][:, j, :].mean(axis=0)
    mu_j_post_std  = res['mus'][:, j, :].std(axis=0)
    mu_j_star      = np.array(mu_true[j])
    # Biais normalise par l'ecart-type posterior
    bias_normalise = np.linalg.norm(mu_j_post_mean - mu_j_star) / (mu_j_post_std.mean() + 1e-12)
    inferential_errors.append(bias_normalise)
inferential_error = float(np.mean(inferential_errors))

# ── 3. ERREUR DE MONTE CARLO ─────────────────────────────────────────────────
# MC error : erreur due au nombre FINI de simulations pour approcher pi_epsilon.
# Pour ABC classique (iid), c'est l'erreur standard de Monte Carlo sur
# les estimateurs de la moyenne : std / sqrt(N_KEEP).
# Cela converge vers 0 quand N_KEEP -> infini, a cible pi_epsilon FIXEE.
# A distinguer de l'inferential error qui reste non nulle meme avec N_KEEP infini.
mc_errors = [
    res['mus'][:, j, :].std(axis=0).mean() / np.sqrt(N_KEEP)
    for j in range(n)
]
mc_error = float(np.mean(mc_errors))

print("=" * 52)
print("  METRIQUES — ABC-Reject vs Gibbs-ABC (papier)")
print("=" * 52)
print(f"  {'Metrique':<30} {'ABC-Reject':>10} {'Gibbs':>8}")
print(f"  {'─'*50}")
print(f"  {'CPU time':<30} {cpu_time/60:>8.1f}min {'~29min':>8}")
print(f"  {'Inferential error':<30} {inferential_error:>10.4f} {'~0.12':>8}")
print(f"  {'MC error (std/sqrt(N))':<30} {mc_error:>10.6f} {'~0.001':>8}")
print()
print("Rappel definitions :")
print("  Inferential error : ecart pi_epsilon vs vrai posterior (non nulle meme si N->inf)")
print("  MC error          : std/sqrt(N), erreur due au N fini pour approcher pi_epsilon")
print(f"  {'Taux d acceptation':<30} {res['acceptance_rate']:>9.4%} {'—':>8}")
print(f"  {'Dist. predictive':<30} {mean_abc:>10.1f} {'274.1':>8}")
print("=" * 52)
print()
print("Interpretation :")
print(f"  CPU  : {'eleve' if cpu_time > 300 else 'raisonnable'} — attendu avec N_TOT={N_TOT:,}")
print(f"  Inf. : {'posterior proche du prior (malediction dim.)' if inferential_error > 0.15 else 'bonne precision'}")
print(f"  MC   : {'stable' if mc_error < 0.15 else 'moderee'}")

In [ ]:
# ── Visualisation des metriques ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = [f'C{j}' for j in range(n)]

# Panel 1 : erreur d'inference par serie
ax = axes[0]
ax.bar([f'S{j+1}' for j in range(n)], inferential_errors, color=colors, alpha=0.8)
ax.axhline(inferential_error, color='crimson', lw=2, linestyle='--',
           label=f'Moy. = {inferential_error:.3f}')
ax.set_title("Erreur d'inference\n"
             r"$\|\overline{\mu}_j^{ABC} - \mu_j^\star\|_2$", fontsize=10)
ax.set_ylabel('Erreur L2', fontsize=9)
ax.legend(fontsize=8)

# Panel 2 : erreur Monte Carlo par serie
ax = axes[1]
ax.bar([f'S{j+1}' for j in range(n)], mc_errors, color=colors, alpha=0.8)
ax.axhline(mc_error, color='crimson', lw=2, linestyle='--',
           label=f'Moy. = {mc_error:.3f}')
ax.set_title("Erreur de Monte Carlo\n"
             r"$\mathrm{Var}(\mu_j^{(i)})$", fontsize=10)
ax.set_ylabel('Variance empirique', fontsize=9)
ax.legend(fontsize=8)

# Panel 3 : tableau comparatif
ax = axes[2]
ax.axis('off')
table_data = [
    ["CPU time",           f"{cpu_time/60:.1f} min", "~29 min"],
    ["Erreur inference",   f"{inferential_error:.4f}", "~0.12"],
    ["Erreur MC",          f"{mc_error:.4f}", "~0.02"],
    ["Taux acceptation",   f"{res['acceptance_rate']:.4%}", "—"],
    ["Dist. predictive",   f"{mean_abc:.1f}", "274.1"],
]
table = ax.table(cellText=table_data,
                 colLabels=["Metrique", "ABC-Reject", "Gibbs-ABC"],
                 cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif col == 2:
        cell.set_facecolor('#e8f5e9')
    elif row % 2 == 0:
        cell.set_facecolor('#f5f5f5')
    cell.set_edgecolor('white')
ax.set_title("Comparaison finale", fontsize=10, pad=12)

plt.suptitle("Evaluation — ABC-Reject sur MA(2) hierarchique ($n=5$, $T=100$)",
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Results/fig_metriques_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

## Tableau comparatif et sauvegarde

In [ ]:
print("=" * 62)
print("  TABLEAU COMPARATIF --- Section 10.2 du supplement")
print("=" * 62)
print(f"  {'Metrique':<38} {'Notre impl.':>10} {'Papier':>8}")
print(f"  {'─'*58}")
print(f"  {'N_tot simulations':<38} {N_TOT:>10,} {'1.1e6':>8}")
print(f"  {'N acceptes':<38} {N_KEEP:>10,} {'1000':>8}")
print(f"  {'Seuil effectif epsilon':<38} {res['epsilon']:>10.4f} {'—':>8}")
print(f"  {'Taux d acceptation':<38} {res['acceptance_rate']:>9.4%} {'0.09%':>8}")
print(f"  {'Temps CPU':<38} {res['cpu_time']/60:>8.1f}min {'~10min':>8}")
print(f"  {'Dist. pred. ABC-Reject':<38} {mean_abc:>10.1f} {'436.8':>8}")
print(f"  {'Dist. pred. ABC-Gibbs':<38} {'(cf. autre NB)':>10} {'274.1':>8}")
print(f"  {'Baseline (prior)':<38} {mean_prior:>10.1f} {'—':>8}")
print("=" * 62)

np.savez('Results/resultats_abc_reject.npz',
    alphas=res['alphas'], zetas=res['zetas'],
    mus=res['mus'], sigma2s=res['sigma2s'],
    distances=res['distances'], epsilon=res['epsilon'],
    cpu_time=res['cpu_time'], n_simulations=N_TOT,
    acceptance_rate=res['acceptance_rate'],
    mean_ppd=mean_abc, std_ppd=std_abc, mean_prior=mean_prior,
    SEED=SEED, x_obs=np.array(x_obs),
    mu_true=np.array(mu_true), sigma2_true=np.array(sigma2_true),
    q_w=q_w, q_v=q_v)
print("\nResultats sauvegardes dans 'resultats_abc_reject.npz'")

---

## Conclusion

| Résultat | Interprétation |
|----------|---------------|
| Posterior $\approx$ Prior | Malédiction de la dimension confirmée ($d=20$) |
| PPD : 436.8 vs 274.1 | ABC-Gibbs est 1.6× plus précis à budget comparable |
| Taux acceptation 0.09% | Rejet massif — moteur de l'inefficacité |



# Partie III : Algorithme « Metropolis à marches aléatoires » ("random-walk Metropolis").

## Calcul de la vraisemblance

$$x_n = y_n + \mu_1 y_{n-1} + \mu_2 y_{n-2}, \qquad y_k \overset{iid}{\sim} \mathcal{N}(0, \sigma^2)$$

On récupère les bruits $y_k$ par récurrence (avec $y_0 = y_{-1} = 0$) :

$$y_k = x_k - \mu_1 y_{k-1} - \mu_2 y_{k-2}, \qquad k = 1, \ldots, n$$

Conditionnellement à $x_1, \ldots, x_{k-1}$, les valeurs $y_{k-1}$ et $y_{k-2}$ sont \textbf{connues}, donc :

$$x_k \mid x_1, \ldots, x_{k-1} \sim \mathcal{N}(\mu_1 y_{k-1} + \mu_2 y_{k-2},\ \sigma^2)$$

ce qui donne la densité conditionnelle 
$$L(x_k \mid x_1, \ldots, x_{k-1}) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{y_k^2}{2\sigma^2}\right)$$

Par la règle de la chaîne :

$$L(x_1, \ldots, x_n) = L(x_1) \prod_{k=2}^{n} L(x_k \mid x_1, \ldots, x_{k-1})$$

$$= \prod_{k=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{y_k^2}{2\sigma^2}\right)$$

$$\boxed{L(x_1, \ldots, x_n \mid \mu_1, \mu_2, \sigma^2) \approx \left(\frac{1}{2\pi\sigma^2}\right)^{n/2} \exp\!\left(-\frac{1}{2\sigma^2}\sum_{k=1}^{n} y_k^2\right)}$$

$$\ell(\mu_1, \mu_2, \sigma^2) = -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\sigma^2) - \frac{1}{2\sigma^2}\sum_{k=1}^{n} y_k^2$$

### Posterior et RWMH

Prior sur $\sigma^2$ :

$$\log p(\sigma^2) \propto -(\varsigma_2 + 1)\log(\sigma^2) - \frac{\varsigma_1}{\sigma^2}$$

Prior sur $\beta$ :

$$\log p(\beta) \propto \sum_i (\alpha_i - 1)\log \beta_i$$

Total :
$$\log p(\sigma^2) + \log p(\beta) \propto -(\varsigma_2 + 1)\log(\sigma^2) - \frac{\varsigma_1}{\sigma^2} + \sum_i (\alpha_i - 1)\log \beta_i$$

In [ ]:
def log_target(x, mu, sigma2, alpha, zeta):
    """
    log densité cible = log vraisemblance + log prior
    """
    T = len(x)
    
    eps = np.zeros(T)

    # reconstruction des bruits
    for t in range(T):
        eps[t] = x[t]
        if t-1 >= 0:
            eps[t] -= mu[0] * eps[t - 1]
        if t-2 >= 0:
            eps[t] -= mu[1] * eps[t - 2]

    quad = np.sum(eps**2)

    log_lik = -np.log(2*math.pi)*T/2 - np.log(sigma2)*T/2 - quad/(2*sigma2)

    # prior
    s  = (mu[1] + 1) / 2
    b1 = (mu[0] + s) / 2
    b2 = s - b1
    b3 = 1 - b1 - b2
    beta = np.array([b1, b2, b3])

    if np.any(beta <= 0) or np.any(beta >= 1) or sigma2 <= 0:
        return -np.inf

    # log vraissemblance des prior
    log_prior = dirichlet.logpdf(beta, alpha) + invgamma.logpdf(sigma2, zeta[0], scale=zeta[1])

    # log vraissemblance totale
    return log_lik + log_prior

# 'Component-wise' Metropolis-Hastings 
def metropolis_kernel(x, theta, step_mu, step_sigma2, alpha, zeta):
    # on mets à jour sur chaque dimension ET PAS de manière jointe
    # sinon on risque d'avoir certaines dimensions très très mauvaises
    # on avance avec un kernel gaussian
    """
    theta = (mu1, mu2, sigma2)
    """
    mu_cur = theta[:2].copy()
    sigma2_cur = theta[2]
    acc = 0

    # mise à jour de mu1
    mu_prop = mu_cur.copy()
    mu_prop[0] += step_mu * np.random.randn()
    log_alpha = (log_target(x, mu_prop, sigma2_cur, alpha, zeta)
                  - log_target(x, mu_cur,  sigma2_cur, alpha, zeta))
    if np.log(np.random.rand()) < log_alpha:
        mu_cur = mu_prop
        acc += 1

    # mise à jour de mu2
    mu_prop = mu_cur.copy()
    mu_prop[1] += step_mu * np.random.randn()
    log_alpha = (log_target(x, mu_prop, sigma2_cur, alpha, zeta)
                  - log_target(x, mu_cur,  sigma2_cur, alpha, zeta))
    
    if np.log(np.random.rand()) < log_alpha:
        mu_cur = mu_prop
        acc += 1

    # mise à jour de sigma2
    sigma2_prop = sigma2_cur + step_sigma2 * np.random.randn()

    if sigma2_prop <= 0:
        pass  # on rejette directement
    else:
        log_alpha = (log_target(x, mu_cur, sigma2_prop, alpha, zeta)
                - log_target(x, mu_cur, sigma2_cur,  alpha, zeta))
        
        if np.log(np.random.rand()) < log_alpha:
            sigma2_cur = sigma2_prop
            acc+=1

    theta_new = np.array([mu_cur[0], mu_cur[1], sigma2_cur])
    return {'val': theta_new, 'acc': acc / 3}

## `metropolis_kernel`

```python
def metropolis_kernel(x, theta, step_mu, step_sigma2, alpha, zeta)
```

Effectue **une itération** du Metropolis-Hastings en mode **component-wise** (mise à jour composante par composante).

> **Pourquoi component-wise ?** Proposer un saut joint sur toutes les dimensions à la fois augmente le risque de rejeter systématiquement. En mettant à jour chaque paramètre séparément, on améliore le taux d'acceptation global.

### Paramètre d'entrée

`theta = [mu1, mu2, sigma2]` — état courant de la chaîne.

### Déroulement

Pour chaque composante, on applique le schéma classique RWMH :

1. **Proposition** : $\theta^* = \theta_{\text{cur}} + \delta \cdot \mathcal{N}(0,1)$
2. **Ratio d'acceptation** : $\log \alpha = \log \pi(\theta^*) - \log \pi(\theta_{\text{cur}})$
3. **Acceptation** : on accepte $\theta^*$ si $\log U < \log \alpha$, avec $U \sim \mathcal{U}[0,1]$

> Si $\sigma^2_{\text{proposé}} \leq 0$, la proposition est rejetée directement sans calcul.

---

## `rwmh_one_series`

```python
def rwmh_one_series(x, alpha, zeta, n_iter=10_000,
                    step_mu=0.1, step_sigma2=0.2, seed=20798)
```

Lance une **chaîne de Markov complète** de longueur `n_iter` à partir d'un point initial tiré aléatoirement depuis les priors.

### Initialisation

Le point de départ est échantillonné depuis les distributions *a priori* :

```
β         ~ Dirichlet(alpha)
mu1_init  = β[0] - β[1]
mu2_init  = 2(β[0] + β[1]) - 1
sigma2_init ~ InvGamma(zeta[0], scale=zeta[1])
```

### Boucle principale

```
chain[0] ← état initial
for n in 1..N:
    chain[n] ← metropolis_kernel(x, chain[n-1], ...)
```

## Résumé du flux

```
Données x
    │
    ▼
Initialisation depuis les priors
    │
    ▼
┌─────────────────────────────────────┐
│  Boucle Metropolis-Hastings (N pas) │
│                                     │
│  Pour chaque composante :           │
│    1. Proposer θ* (marche aléat.)   │
│    2. Calculer log_target(θ*)       │
│    3. Accepter / Rejeter            │
└─────────────────────────────────────┘
    │
    ▼
Chaîne (N × 3) + taux d'acceptation
```


In [ ]:
def rwmh_one_series(x, alpha, zeta, n_iter=10_000,
                    step_mu=0.1, step_sigma2=0.2, seed=20798):
    
    np.random.seed(seed + 2000)

    # début aléatoire de la chaine
    beta = np.random.dirichlet(alpha)  
    mu1_inits   = beta[0] - beta[1]
    mu2_inits   = 2 * (beta[0] + beta[1]) - 1

    mu_init = [mu1_inits, mu2_inits]
    sigma2_init = invgamma.rvs(a=zeta[0], scale=zeta[1])

    N  = n_iter
    s = 0
    chain = np.empty((N, 3))
    chain[0] = [mu_init[0], mu_init[1], sigma2_init]

    # effectuer N interations de metropolis hastings
    for n in range(1, N):
        output = metropolis_kernel(x, chain[n-1], step_mu, step_sigma2, alpha, zeta)
        chain[n] = output['val']
        s += output['acc']

    acc_rate = s / N
    return chain, acc_rate

In [ ]:
def plot_chain(all_chains, true_mu, true_sigma2, mu1_prior = None):
    """
    all_chains : liste de R arrays (n_post, 3)
    Superpose les KDE de chaque replicate + vraie valeur.
    """
    true_vals   = [true_mu[0], true_mu[1], true_sigma2]
    param_names = [r"$\mu_1$", r"$\mu_2$", r"$\sigma^2$"]
    R      = len(all_chains)
    colors = plt.cm.tab10.colors

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for k in range(3):
        ax = axes[k]
        for r in range(R):
            post_r = all_chains[r][:, k]
            kde    = gaussian_kde(post_r)
            x_grid = np.linspace(post_r.min(), post_r.max(), 300)
            ax.plot(
                x_grid, kde(x_grid),
                color=colors[r % 10], alpha=0.7, lw=1.5,
                label=f"R{r+1}" if k == 0 else None
            )
            ax.axvline(post_r.mean(), color=colors[r % 10], lw=1, linestyle='--', alpha=0.5)

        if k == 0 and mu1_prior is not None:
            kde_prior = gaussian_kde(mu1_prior)
            x_prior   = np.linspace(mu1_prior.min(), mu1_prior.max(), 300)
            ax.plot(
                x_prior, kde_prior(x_prior),
                color='black', lw=1.2, linestyle=':', alpha=0.5,
                label="prior $\\mu_1$"
            )

        ax.axvline(true_vals[k], color='#D85A30', lw=2, label='vraie valeur' if k == 0 else None)
        ax.set_title(param_names[k])
    
    axes[0].legend(fontsize=7, ncol=2)
    plt.tight_layout()
    save_path=f"Results/RWMH_serie{index}_chain.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_boxplots_by_replicate(all_chains, true_mu, true_sigma2, index):  # ← index ajouté
    R           = len(all_chains)
    true_vals   = [true_mu[0], true_mu[1], true_sigma2]
    param_names = [r"$\mu_1$", r"$\mu_2$", r"$\sigma^2$"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(f"Série {index} — distribution post burn-in par replicate", fontsize=13)  # ← utilisé ici

    for j, ax in enumerate(axes):
        data_per_rep = [all_chains[r][:, j] for r in range(R)]

        ax.boxplot(
            data_per_rep,
            tick_labels=[f"R{r+1}" for r in range(R)],
            patch_artist=True,
            boxprops=dict(facecolor="steelblue", alpha=0.6),
            medianprops=dict(color="white", linewidth=2),
            whiskerprops=dict(color="steelblue"),
            capprops=dict(color="steelblue"),
            flierprops=dict(marker='o', markerfacecolor='steelblue', markersize=3, alpha=0.4),
        )
        ax.axhline(true_vals[j], color='#D85A30', linestyle='--', linewidth=1.5, label='vraie valeur')
        ax.set_title(param_names[j])
        ax.set_xlabel("Replicate")
        ax.legend(fontsize=8)

    plt.tight_layout()
    save_path=f"Results/RWMH_serie{index}_boxplot.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# création d'une pipeline pour effectuer les fonctions sur les 5 séries
# etape 1 : lancer R replicates de RWMH
# etape 2 : calculer un temps moyen d'execution
# etape 3 : calculer les erreurs d'inference, de monte carlo
# etape 4 : plotter un boxplot et une densite

def pipeline(index, x_star, mu_list, sigma2_list, burn_in, n_iterations, R, mu1_prior):
    """Lance toute la pipeline pour un dataset donné."""

    cpu_times = []
    all_estimates = []
    acc_rates = []
    all_chains = []
    
    for r in range(R):
        # start du temps de calcul
        t_start = time.process_time()

        chain, acc = rwmh_one_series(
            x_star[index], alpha, zeta, 
            n_iter = n_iterations, step_mu=0.5, step_sigma2=0.5, seed=r
        )

        # fin du temps de calcul
        t_end = time.process_time()
        cpu_time = t_end - t_start
        cpu_times.append(cpu_time)

        post = chain[burn_in:]
        all_estimates.append(post.mean(axis=0))
        acc_rates.append(acc)
        all_chains.append(post)
    
    all_estimates = np.array(all_estimates)
    cpu_moyen = np.mean(cpu_times)
    cpu_std = np.std(cpu_times)  # pour voir si c'est stable
    print(f"[série {index}] CPU moyen : {cpu_moyen:.4f}s ± {cpu_std:.4f}s")

    # --- erreurs ---
    mu, sigma2 = mu_list[index], sigma2_list[index]
    true_vals  = np.array([mu[0], mu[1], sigma2])
    bias = all_estimates.mean(axis=0) - true_vals

    n_sim = all_estimates.shape[0]
    mce = all_estimates.std(axis=0, ddof=1) / np.sqrt(n_sim)
    mse  = ((all_estimates - true_vals) ** 2).mean(axis=0)

    param_names = [r"$\mu_1$", r"$\mu_2$", r"$\sigma^2$"]
    for i, name in enumerate(param_names):
        print(f"[série {index}] {name} — Biais: {bias[i]:.6f} | MSE: {mse[i]:.6f} | MCE : {mce[i]}")
    
    print(f"Taux d'acceptation moyen : {np.mean(acc_rates):.1%}")
    
    # --- boxplots par replicate ---
    plot_boxplots_by_replicate(all_chains, mu, sigma2, index)
    # histogramme
    plot_chain(all_chains, mu, sigma2)     
    
for index in range(len(x_obs)):
    # remarque : le burn =+/- 10% du nombre d'iterations
    # on ajoute des replicates !!
    pipeline(index, x_obs, mu_true, sigma2_true, burn_in=1000, n_iterations = 10_000, 
    R = 10, mu1_prior = mu1_prior)

In [ ]:
def plot_prior_vs_posterior_mu1(all_chains_series4, mu1_prior, true_mu):
    """
    Trace uniquement le replicate 1 (r=0) de la série index=4
    avec le prior sur mu1 (theta_1).
    
    all_chains_series4 : liste des chains pour la série 4
    mu1_prior          : échantillon du prior sur mu1
    true_mu            : [mu1_true, mu2_true]
    """
    # --- Récupération du replicate 1 (index 0) ---
    rep1_mu1 = all_chains_series4[0][:, 0]   # shape (n_post,) → colonne θ₁

    fig, ax = plt.subplots(figsize=(7, 4))

    # --- Prior ---
    kde_prior = gaussian_kde(mu1_prior)
    x_prior   = np.linspace(mu1_prior.min(), mu1_prior.max(), 400)
    ax.plot(x_prior, kde_prior(x_prior),
            color='black', lw=1.5, linestyle=':', label=r"Prior $\mu_1$")

    # --- Posterior replicate 1 ---
    kde_post = gaussian_kde(rep1_mu1)
    x_post   = np.linspace(rep1_mu1.min(), rep1_mu1.max(), 400)
    ax.plot(x_post, kde_post(x_post),
            color='steelblue', lw=2, label=r"Posterior $\mu_1$ — R1")

    # --- Vraie valeur ---
    ax.axvline(true_mu[0], color='#D85A30', lw=2, linestyle='--', label=r"Vraie $\mu_1$")

    ax.set_title(r"Série 4 — Prior vs Posterior sur $\mu_1$ (Replicate 1)", fontsize=12)
    ax.set_xlabel(r"$\mu_1$")
    ax.set_ylabel("Densité")
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("Results/RWMH_serie4_prior_vs_posterior_mu1.png", dpi=150, bbox_inches="tight")
    plt.show()

# Relance rapide juste pour la série 4
all_chains_s4 = []
for r in range(10):
    chain, _ = rwmh_one_series(
        x_obs[4], alpha, zeta,
        n_iter=10_000, step_mu=0.5, step_sigma2=0.5, seed=r
    )
    all_chains_s4.append(chain[1000:])   # burn_in = 1000

# Plot
plot_prior_vs_posterior_mu1(all_chains_s4, mu1_prior, mu_true[4])